# XN Benchmark: BioMiner and MolScribe on Colab

Runs the full pipeline from this repo on a Colab GPU runtime: setup, inference, metrics, and results.
The BioVista images and Jirapa's DECIMER results are downloaded automatically.

**Before you start**
- Runtime: GPU. An L4 or A100 is recommended, since BioMiner runs three model servers at once.
- Google Drive folder (set `DATA_DIR` below) containing:
  - `course_papers/` with the six course PDFs and the ground truth CSV from Canvas
  - `decimer_crops/` with Jirapa's DECIMER crops of those papers (if missing, only that step is skipped)
- Colab secrets (key icon in the left sidebar): `BIOMINER_API_KEY`, `BIOMINER_BASE_URL`,
  `BIOMINER_TEXT_MODEL`, `BIOMINER_VISION_MODEL`

## 1. Install conda
The runtime restarts automatically after this cell. Then continue with the next cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Settings
The only two values to edit.

In [ ]:
REPO_URL = "https://github.com/<your-username>/xn-ocsr-benchmark.git"
DATA_DIR = "/content/drive/MyDrive/xn_data"

## 3. Get the code and link the data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!git clone -q {REPO_URL} /content/xn-ocsr-benchmark
%cd /content/xn-ocsr-benchmark
!ln -sfn {DATA_DIR} data
!ls data

## 4. Download BioMiner, the BioVista data, model weights, and Jirapa's DECIMER results

In [ ]:
!pip install -q -r requirements.txt
!python scripts/setup_external.py

## 5. Create one environment per tool
The exact package versions are exported to `envs/lock/` for reproducibility.

In [ ]:
!sed -i '/^prefix:/d' external/BioMiner/main_environment.yml  # drop the authors' local install path
!conda env create -q -n BioMiner -f external/BioMiner/main_environment.yml
!conda env create -q -f envs/molscribe.yml
!conda env create -q -f envs/scoring.yml

!mkdir -p envs/lock
!conda env export -n BioMiner > envs/lock/BioMiner.lock.yml
!conda env export -n xn-molscribe > envs/lock/xn-molscribe.lock.yml
!conda env export -n xn-scoring > envs/lock/xn-scoring.lock.yml

# MinerU's config and models, inside the BioMiner environment
!python scripts/setup_external.py --steps mineru

## 6. Load API credentials from Colab secrets
Keys are read into environment variables only. They are never written into the repo.

In [ ]:
import os
from google.colab import userdata

for name in ["BIOMINER_API_KEY", "BIOMINER_BASE_URL", "BIOMINER_TEXT_MODEL", "BIOMINER_VISION_MODEL"]:
    os.environ[name] = userdata.get(name)

## 7. Check the scoring code

In [ ]:
!conda run -n xn-scoring python -m pytest -q tests

## 8. Run the full pipeline
MolScribe on the BioVista images, MolScribe on the course crops, BioMiner on the course papers, BioMiner on the BioVista subset, then evaluation.

In [ ]:
!python run_pipeline.py

## 9. Results

In [ ]:
!cat results/summary.md

## 10. Download results to commit them
Upload the contents to the repo's `results/` and `envs/lock/` folders.

In [ ]:
!zip -qr results_and_locks.zip results envs/lock -x "results/raw/*"
from google.colab import files
files.download("results_and_locks.zip")